# Marswalk — rock / boulder detector training (REAL DATA)

Trains a boulder **detector** on the real **BOULDERING** dataset
(Prieur et al., 2023) and evaluates it with the Marswalk metric
(`ml.metrics_det`, **F1 by size**, gate ≥ 1.5 m) and the spatial holdout
(`ml.splits`). **No synthetic data** — you download the real labelled imagery
and train on it.

| Dataset | DOI | License | Use |
|---|---|---|---|
| **BOULDERING** (images + boulder labels + Detectron2 JSON + pretrained model) | [10.5281/zenodo.8171052](https://doi.org/10.5281/zenodo.8171052) | **CC BY 4.0** | **primary** labelled set (orbital boulders) |
| BOULDERING raw (rasters + shapefiles) | same record | CC BY 4.0 | full-resolution rasters (9.4 GB) |
| AI4Mars (rover-view terrain) | [10.5281/zenodo.15995036](https://doi.org/10.5281/zenodo.15995036) | **CC BY 4.0** on Zenodo | optional; rover-view, see note |

**Attribution (required by CC BY 4.0):** cite Prieur, N. C., Amaro, B., Gonzalez,
E., Kerner, H., et al., *BOULDERING* dataset, Zenodo, 2023,
doi:10.5281/zenodo.8171052. Code/tools: github.com/astroNils (MLtools,
rastertools, shptools, YOLOv8-BeyondEarth).

### Honest status — read before trusting any number
- **Run this on the owner machine (RTX 4050 + the dataset download).** The
  Marswalk sandbox has no GPU and does not download the dataset; the data→label
  →train→eval *code* was verified, the *trained result is not yet produced here*.
- **AI4Mars licence note:** `DATA_LICENSES.md` says "unspecified on the NASA
  portal"; the **Zenodo** record 15995036 states **CC BY 4.0**. Confirm which is
  authoritative before shipping AI4Mars-derived products. Also AI4Mars is
  *rover-view* navcam terrain (semantic classes incl. "big rock"), not
  map-registered orbital boulders — BOULDERING fits map-based routing better.
- **Gate (MARSWALK_CONTEXT.md §3):** the detector informs routing only after the
  P6 ablation shows rock-aware routes reduce rock exposure vs slope-only. Until
  then this is a capability, not a validated routing input.
- The diameter-based size bins need a real **ground sample distance (GSD)**; the
  loader reads it from each raster (rasterio). If GSD is unknown it warns and
  reports pixels — it never invents a scale.

## 0. Setup

In [ ]:
import sys, pathlib, time, json, hashlib, urllib.request, zipfile
import numpy as np
import torch, torchvision
from torch.utils.data import DataLoader, Dataset

# make marswalk importable (or `pip install -e .`)
for _c in (pathlib.Path.cwd() / "src", pathlib.Path.cwd().parent / "src"):
    if (_c / "marswalk").exists() and str(_c) not in sys.path:
        sys.path.insert(0, str(_c)); break

from marswalk.ml.annotation import AnnotationSet, BBox, LabelSource, RockLabel, Tile
from marswalk.ml.metrics_det import precision_recall_by_size
from marswalk.ml.splits import spatial_block_split, splits_are_disjoint  # optional

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| device", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no CUDA — training will be slow; use the RTX 4050.")

## 1. Dataset config — real Zenodo files (DOI 10.5281/zenodo.8171052)

In [ ]:
# Verified from the Zenodo REST API (sizes in bytes, md5 as published).
BOULDERING = {
    "record": "8171052",
    "files": {
        "masks": {  # 265 MB: preprocessed patches + labels + Detectron2 JSON (recommended)
            "name": "Apr2023-Mars-Moon-Earth-mask-5px.zip",
            "url": "https://zenodo.org/api/records/8171052/files/Apr2023-Mars-Moon-Earth-mask-5px.zip/content",
            "md5": "af7ae52817d0f42cdda9d10260aa49f9", "size": 265_060_000},
        "model": {  # 655 MB: authors' pretrained Mask R-CNN + config + json
            "name": "best_model.zip",
            "url": "https://zenodo.org/api/records/8171052/files/best_model.zip/content",
            "md5": "6208aa6f231ae046764b05a8868732c0", "size": 655_000_000},
        "raw": {    # 9.4 GB: full rasters + boulder shapefiles
            "name": "raw_data_BOULDERING.zip",
            "url": "https://zenodo.org/api/records/8171052/files/raw_data_BOULDERING.zip/content",
            "md5": "ad8592ada710a00ec2cd8e0db8be9896", "size": 9_386_600_000},
    },
    # the masks zip extracts to this subdir (verified from the real archive):
    "root_subdir": "Apr2023-Mars-Moon-Earth-mask-5px",
    # Detectron2 custom-dataset JSON (authors recommend this one):
    "json_rel": "Apr2023-Mars-Moon-Earth-mask-5px/json/Apr2023-Mars-Moon-Earth-mask-5px.json",
}

# Body is encoded in the file_name prefix (there is no body field; `dataset` holds
# the split). Mars = HiRISE ESP_/PSP_ (49 patches); Moon = LROC M*/NAC_ (656);
# Earth = place names (~248). The dataset is designed for CROSS-BODY transfer, so
# training on all bodies and evaluating on Mars is the intended use.
BODY_PREFIXES = {"mars": ("ESP", "PSP"), "moon": ("M", "NAC"),
                 "earth": ("courtright", "stonegarden", "southdeadman")}
BODY_FILTER = None     # None = all bodies (recommended); "mars" = HiRISE only (49 patches)
DEFAULT_PX_SIZE_M = None  # leave None -> read GSD from the .tif; set a float only if you KNOW it
DATA_DIR = pathlib.Path("data/bouldering")   # gitignored; put the download here
print("download target:", DATA_DIR.resolve())

## 2. Download the real dataset (owner step)

Downloads the 265 MB masks/JSON bundle and verifies the MD5. Set `GET_MODEL` /
`GET_RAW` to also pull the pretrained model (655 MB) or the full rasters
(9.4 GB). Re-running skips files already present with a matching checksum.

In [ ]:
GET_MASKS, GET_MODEL, GET_RAW = True, False, False

def _md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def download(spec, dest_dir):
    dest_dir.mkdir(parents=True, exist_ok=True)
    out = dest_dir / spec["name"]
    if out.exists() and _md5(out) == spec["md5"]:
        print("ok (cached):", spec["name"]); return out
    print(f"downloading {spec['name']} (~{spec['size']/1e6:.0f} MB) ...")
    t0 = time.time()
    with urllib.request.urlopen(spec["url"]) as r, open(out, "wb") as f:
        total, done = spec["size"], 0
        while True:
            buf = r.read(1 << 20)
            if not buf:
                break
            f.write(buf); done += len(buf)
            if done % (50 << 20) < (1 << 20):
                print(f"  {done/1e6:6.0f}/{total/1e6:.0f} MB", end="\r")
    got = _md5(out)
    if got != spec["md5"]:
        raise RuntimeError(f"MD5 mismatch for {spec['name']}: {got} != {spec['md5']}")
    print(f"\nverified {spec['name']} in {time.time()-t0:.0f}s")
    return out

to_get = [k for k, on in [("masks", GET_MASKS), ("model", GET_MODEL), ("raw", GET_RAW)] if on]
paths = {k: download(BOULDERING["files"][k], DATA_DIR) for k in to_get}

def unzip(zpath, dest):
    with zipfile.ZipFile(zpath) as z:
        z.extractall(dest)
    print("extracted", zpath.name)

if GET_MASKS:
    unzip(paths["masks"], DATA_DIR)

## 3. Load the real labels (Detectron2 JSON → Marswalk `AnnotationSet`)

In [ ]:
# Detectron2 BoxMode ints (detectron2.structures.BoxMode)
XYXY_ABS, XYWH_ABS = 0, 1

def _to_xywh(bbox, mode):
    if mode == XYWH_ABS:
        x, y, w, h = bbox
    elif mode == XYXY_ABS:
        x1, y1, x2, y2 = bbox; x, y, w, h = x1, y1, x2 - x1, y2 - y1
    else:
        raise ValueError(f"bbox_mode {mode}: only 0=XYXY_ABS / 1=XYWH_ABS handled")
    return float(x), float(y), float(w), float(h)

def _body_ok(fn, flt):
    if not flt:
        return True
    return fn.upper().startswith(tuple(p.upper() for p in BODY_PREFIXES[flt]))

def _img_path(root, split, fn):
    return pathlib.Path(root) / BOULDERING["root_subdir"] / "preprocessing" / split / "images" / fn

def _read_gsd_m(img_path):
    # Real ground sample distance from the .tif sibling; None if unreadable/ungeoref.
    tif = pathlib.Path(str(img_path).replace("_image.png", "_image.tif"))
    try:
        import rasterio
        with rasterio.open(tif) as ds:
            g = abs(ds.transform.a)
            return g if (ds.crs is not None and g > 0) else None
    except Exception:
        return None

def load_bouldering(data_root, body_filter=BODY_FILTER, default_px=DEFAULT_PX_SIZE_M):
    jp = pathlib.Path(data_root) / BOULDERING["json_rel"]
    recs = json.loads(jp.read_text(encoding="utf-8"))
    recs = recs["dataset"] if isinstance(recs, dict) and "dataset" in recs else recs
    print("first record keys:", sorted(recs[0].keys()))  # sanity-check vs the real file
    tiles, splits, no_gsd = [], {"train": [], "test": []}, 0
    for rec in recs:
        fn = str(rec.get("file_name", ""))
        if not _body_ok(fn, body_filter):
            continue
        split = str(rec.get("dataset", "train"))          # 'dataset' field = official split
        rid = fn                                           # file_name is unique; image_id is not
        img_path = _img_path(data_root, split, fn)
        gsd = _read_gsd_m(img_path) or default_px
        if gsd is None:
            no_gsd += 1; gsd = 1.0  # placeholder; metres NOT meaningful (warned below)
        labels = []
        for a in rec.get("annotations", []):
            x, y, w, h = _to_xywh(a["bbox"], a.get("bbox_mode", XYXY_ABS))
            if w > 0 and h > 0:
                labels.append(RockLabel(bbox=BBox(x=x, y=y, w=w, h=h), source=LabelSource.MANUAL))
        tiles.append(Tile(id=rid, path=str(img_path), px_size_m=gsd,
                          width=int(rec.get("width", 0)) or 1,
                          height=int(rec.get("height", 0)) or 1, labels=labels))
        (splits["train"] if split == "train" else splits["test"]).append(rid)  # val+test -> test
    ann = AnnotationSet(tiles=tiles)
    print(f"tiles: {len(ann.tiles)} | boxes: {sum(len(t.labels) for t in ann.tiles)} | "
          f"train {len(splits['train'])} / test {len(splits['test'])}")
    if no_gsd:
        print(f"WARNING: {no_gsd} tiles had no readable GSD (install rasterio; the .tif must be "
              f"georeferenced, or set DEFAULT_PX_SIZE_M per HiRISE product). Metre-size bins are "
              f"not meaningful for those.")
    return ann, splits

annotations, splits = load_bouldering(DATA_DIR)

## 4. Train / test split

The authors' split is the record's `dataset` field (train / validation / test);
`load_bouldering` returns it (validation+test folded into the holdout).
(`ml.splits.spatial_block_split` is available when you have map coordinates and
want a Marswalk spatial holdout instead.)

In [ ]:
train_ids, test_ids = splits["train"], splits["test"]
if not train_ids or not test_ids:
    raise RuntimeError("empty split — check BODY_FILTER (Mars-only is just 49 patches) "
                       "and that the JSON loaded.")
print(f"train {len(train_ids)} / test {len(test_ids)} (authors' split, body={BODY_FILTER or 'all'})")

## 5. Dataset + model (torchvision Faster R-CNN on the real patches)

In [ ]:
def read_image(path):
    # Real patch -> (H,W) float32 in [0,1]. GeoTIFF via rasterio, else PIL.
    try:
        import rasterio
        with rasterio.open(path) as ds:
            arr = ds.read(1).astype("float32")
    except Exception:
        from PIL import Image
        arr = np.asarray(Image.open(path).convert("L"), dtype="float32")
    lo, hi = float(np.nanmin(arr)), float(np.nanmax(arr))
    return (arr - lo) / (hi - lo) if hi > lo else np.zeros_like(arr)

class BoulderDataset(Dataset):
    def __init__(self, ann, ids):
        self.tiles = [ann.by_id(i) for i in ids]
    def __len__(self):
        return len(self.tiles)
    def __getitem__(self, idx):
        t = self.tiles[idx]
        img = read_image(t.path)
        img_t = torch.from_numpy(img).unsqueeze(0).repeat(3, 1, 1)
        boxes = [[l.bbox.x, l.bbox.y, l.bbox.x + l.bbox.w, l.bbox.y + l.bbox.h] for l in t.labels]
        if boxes:
            target = {"boxes": torch.tensor(boxes, dtype=torch.float32),
                      "labels": torch.ones(len(boxes), dtype=torch.int64)}
        else:
            target = {"boxes": torch.zeros((0, 4), dtype=torch.float32),
                      "labels": torch.zeros((0,), dtype=torch.int64)}
        return img_t, target, t.id

def collate(b):
    return tuple(zip(*b, strict=True))

def build_model(num_classes=2, pretrained=True):
    w = "DEFAULT" if pretrained else None
    return torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=w, weights_backbone=w, num_classes=num_classes)

model = build_model().to(device)
print("model ready:", sum(p.numel() for p in model.parameters()) / 1e6, "M params")

## 6. Train — AMP + channels_last + grad accumulation; logs peak VRAM & epoch time

Alternative (closest to the dataset's own setup): load the authors' **best_model**
with Detectron2 (`best_model.zip` has `Base-RCNN-FPN.yaml` + weights + the JSON).
Install detectron2 and register the JSON with `DatasetCatalog.register`; see
github.com/astroNils/MLtools. The torchvision path below avoids the detectron2
build on Windows and trains on the same labels.

In [ ]:
EPOCHS, BATCH, GRAD_ACCUM, LR = 20, 2, 2, 5e-3
AMP = True; CHANNELS_LAST = True

def train(model, ann, ids):
    dl = DataLoader(BoulderDataset(ann, ids), batch_size=BATCH, shuffle=True, collate_fn=collate)
    use_amp = AMP and device.type == "cuda"; cl = CHANNELS_LAST and device.type == "cuda"
    if cl:
        model = model.to(memory_format=torch.channels_last)
    opt = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                          lr=LR, momentum=0.9, weight_decay=5e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    model.train()
    for ep in range(EPOCHS):
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        t0, run = time.time(), 0.0; opt.zero_grad()
        for step, (imgs, targets, _ids) in enumerate(dl):
            imgs = [(im.to(device, memory_format=torch.channels_last) if cl else im.to(device)) for im in imgs]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss = sum(model(imgs, targets).values()) / GRAD_ACCUM
            scaler.scale(loss).backward()
            if (step + 1) % GRAD_ACCUM == 0:
                scaler.step(opt); scaler.update(); opt.zero_grad()
            run += float(loss) * GRAD_ACCUM
        peak = (torch.cuda.max_memory_allocated() / 1e9) if device.type == "cuda" else float("nan")
        print(f"epoch {ep+1}/{EPOCHS} | loss {run/max(len(dl),1):.3f} | {time.time()-t0:.1f}s | peak VRAM {peak:.2f} GB")
    return model

# Tip: set EPOCHS=1 first to confirm the loop + VRAM fit, then raise it.
model = train(model, annotations, train_ids)

## 7. Evaluate on the holdout — F1 by size (gate ≥ 1.5 m)

In [ ]:
SCORE_THR, IOU_THR = 0.5, 0.5

@torch.no_grad()
def evaluate(model, ann, ids):
    model.eval(); agg = {}
    for tid in ids:
        t = ann.by_id(tid)
        img = read_image(t.path)
        img_t = torch.from_numpy(img).unsqueeze(0).repeat(3, 1, 1).to(device)
        pred = model([img_t])[0]
        keep = pred["scores"] >= SCORE_THR
        pboxes = [BBox(x=float(b[0]), y=float(b[1]), w=float(b[2]-b[0]), h=float(b[3]-b[1]))
                  for b in pred["boxes"][keep].cpu()]
        pscores = [float(s) for s in pred["scores"][keep].cpu()]
        res = precision_recall_by_size([*pboxes], [l.bbox for l in t.labels], t.px_size_m,
                                       iou_thr=IOU_THR, scores=pscores)
        for k, v in res.items():
            a = agg.setdefault(k, {"tp": 0.0, "fp": 0.0, "fn": 0.0})
            for m in ("tp", "fp", "fn"):
                a[m] += v[m]
    print(f"{'size bin':>14} | {'P':>5} {'R':>5} {'F1':>5} | tp/fp/fn")
    gate = 0.0
    for k, a in agg.items():
        p = a["tp"]/(a["tp"]+a["fp"]) if (a["tp"]+a["fp"]) else 0.0
        r = a["tp"]/(a["tp"]+a["fn"]) if (a["tp"]+a["fn"]) else 0.0
        f1 = 2*p*r/(p+r) if (p+r) else 0.0
        if k == "[1.5,inf)m":
            gate = f1
        print(f"{k:>14} | {p:.2f} {r:.2f} {f1:.2f} | {a['tp']:.0f}/{a['fp']:.0f}/{a['fn']:.0f}")
    print(f"\nGATE — F1 for boulders >= 1.5 m: {gate:.3f}  (aspirational >= 0.8)")
    return agg

_ = evaluate(model, annotations, test_ids)

## 8. Format self-check (no download needed)

Validates the Detectron2-JSON → `AnnotationSet` conversion on a tiny in-memory
fixture — this is a **code check, not training data**. If it passes, the loader
in §3 is wired correctly; the real result still depends on your downloaded data.

In [ ]:
def _selfcheck():
    fx = [{"file_name": "preprocessing/train/images/mars_P01.tif", "image_id": "mars_P01",
           "height": 512, "width": 512, "annotations": [
               {"bbox": [100, 100, 140, 150], "bbox_mode": XYXY_ABS, "category_id": 0},
               {"bbox": [10, 10, 8, 8], "bbox_mode": XYWH_ABS, "category_id": 0}]},
          {"file_name": "preprocessing/train/images/moon_L02.tif", "image_id": "moon_L02",
           "height": 512, "width": 512, "annotations": [
               {"bbox": [0, 0, 20, 20], "bbox_mode": XYWH_ABS, "category_id": 0}]}]
    tiles = []
    for rec in fx:
        if "mars" not in rec["file_name"]:
            continue
        labs = []
        for a in rec["annotations"]:
            x, y, w, h = _to_xywh(a["bbox"], a["bbox_mode"])
            labs.append(RockLabel(bbox=BBox(x=x, y=y, w=w, h=h)))
        tiles.append(Tile(id=rec["image_id"], px_size_m=0.25, width=512, height=512, labels=labs))
    ann = AnnotationSet(tiles=tiles)
    assert len(ann.tiles) == 1 and len(ann.tiles[0].labels) == 2
    b = ann.tiles[0].labels[0].bbox
    assert (b.x, b.y, b.w, b.h) == (100, 100, 40, 50)
    assert ann.tiles[0].labels[0].diameter_m(0.25) == 12.5
    print("format self-check OK")

_selfcheck()

## 9. What to paste back to the team

After a real run on the RTX 4050:
- [ ] **Peak VRAM** (GB) and **epoch time** (s) at the `BATCH`/`GRAD_ACCUM` used.
- [ ] **F1-by-size table** on the held-out split, especially **F1 ≥ 1.5 m**
      (aspirational ≥ 0.8).
- [ ] Which bodies you trained on (`BODY_FILTER`), tile count, GSD source.
- [ ] Confirm the GSD was read from rasters (metres are real, not placeholder).
- [ ] Any config changes from defaults.

Then update `AI_LOG.md` (P2 row) and `DATA_LICENSES.md` (BOULDERING = CC BY 4.0,
confirmed) with your name/date. **Cite Prieur et al. (2023), doi:10.5281/zenodo.8171052.**
This stays a *capability* until the P6 ablation shows rock-aware cost changes routes.